# Pandas vs Polars Test: BigQuery > Data cleaning > Unsupervised Learning

---


In [1]:
# If you run this in a fresh environment, install deps:
# !pip install -U pandas polars pyarrow google-cloud-bigquery google-cloud-bigquery-storage scikit-learn numpy

import gc
import os
import platform
import time
from dataclasses import dataclass
from typing import Any, Callable, Dict, List

import numpy as np
import pandas as pd
import polars as pl
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("polars:", pl.__version__)

Python: 3.11.1
Platform: macOS-15.5-arm64-arm-64bit
pandas: 3.0.1
polars: 1.38.1


## 1) Configure BigQuery query


In [18]:
PROJECT_ID = 'netflix-user-behavior'
SQL = os.environ.get(
    "BQ_SQL",
    """
    SELECT
    m.*,
    w.*
FROM `netflix-user-behavior.kaggle_uncleaned.movies` AS m
LEFT JOIN `netflix-user-behavior.kaggle_uncleaned.watch_history` AS w
    ON m.movie_id = w.movie_id
LIMIT 1000;
    """.strip()
)

print("PROJECT_ID:", PROJECT_ID)
print("SQL preview:\n", SQL[:300], "..." if len(SQL) > 300 else "")

PROJECT_ID: netflix-user-behavior
SQL preview:
 SELECT
    m.*,
    w.*
FROM `netflix-user-behavior.kaggle_uncleaned.movies` AS m
LEFT JOIN `netflix-user-behavior.kaggle_uncleaned.watch_history` AS w
    ON m.movie_id = w.movie_id
LIMIT 1000; 


In [ ]:
# BigQuery download helpers
# gcloud config set project netflix-user-behavior run in terminal 
from google.cloud import bigquery


def load_bigquery_to_pandas(project_id: str, sql: str) -> pd.DataFrame:
    client = bigquery.Client(project=project_id)
    job = client.query(sql)
    try:
        from google.cloud import bigquery_storage
        bqstorage = bigquery_storage.BigQueryReadClient()
        df = job.result().to_dataframe(bqstorage_client=bqstorage, create_bqstorage_client=False)
    except Exception:
        df = job.result().to_dataframe()
    return df

# Timed load (network + conversion)
t0 = time.perf_counter()
pdf = load_bigquery_to_pandas(PROJECT_ID, SQL)
t1 = time.perf_counter()

print(f"Loaded pandas df: shape={pdf.shape} in {t1-t0:.3f}s")

/Users/allisonpeng/Library/Python/3.11/lib/python/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded pandas df: shape=(1000, 30) in 1.722s


In [ ]:
# Load BigQuery data directly into Polars
import polars as pl


def load_bigquery_to_polars(project_id: str, sql: str) -> pl.DataFrame:
    """Load BigQuery data directly into Polars using Arrow format."""
    client = bigquery.Client(project=project_id)
    job = client.query(sql)
    results = job.result()
    
    # Convert to Arrow table
    arrow_table = results.to_arrow()
    
    # Convert Arrow to Polars DataFrame
    df = pl.from_arrow(arrow_table)
    return df

# Time the direct Polars load
print("Loading BigQuery data directly into Polars...")
t3 = time.perf_counter()
pldf_direct = load_bigquery_to_polars(PROJECT_ID, SQL)
t4 = time.perf_counter()

load_time = t4 - t3
print(f"✓ Loaded Polars DataFrame directly: shape={pldf_direct.shape} in {load_time:.3f}s")
#print(f"  Columns: {list(pldf_direct.columns)}")
# print(f"  Memory usage: {pldf_direct.estimated_size('mb'):.2f} MB")


Loading BigQuery data directly into Polars...


/Users/allisonpeng/Library/Python/3.11/lib/python/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✓ Loaded Polars DataFrame directly: shape=(1000, 30) in 1.301s


In [21]:
# Summary to load in dataframe using pandas vs polars 

print(f"Loaded pandas df in {t1-t0:.3f}s, Loaded polars df in {load_time:.3f}s")
pandas_load = t1 - t0
percent_faster = (pandas_load - load_time) / load_time * 100
print(f"Loading in polars df is {percent_faster:.2f}% faster than pandas")

Loaded pandas df in 1.722s, Loaded polars df in 1.301s
Loading in polars df is 32.32% faster than pandas


## 2) Choose columns & define benchmark workload

---


In [23]:
print(pdf)


       movie_id        title content_type genre_primary genre_secondary  \
0    movie_0140  City Empire  Documentary         Sport             NaN   
1    movie_0140  City Empire  Documentary         Sport             NaN   
2    movie_0140  City Empire  Documentary         Sport             NaN   
3    movie_0140  City Empire  Documentary         Sport             NaN   
4    movie_0140  City Empire  Documentary         Sport             NaN   
..          ...          ...          ...           ...             ...   
995  movie_0328  Mystery Day  Documentary        Horror          Family   
996  movie_0328  Mystery Day  Documentary        Horror          Family   
997  movie_0328  Mystery Day  Documentary        Horror          Family   
998  movie_0328  Mystery Day  Documentary        Horror          Family   
999  movie_0328  Mystery Day  Documentary        Horror          Family   

     release_year  duration_minutes rating language country_of_origin  ...  \
0            2023    

In [ ]:
NUMERIC_COLS = [c for c in pdf.columns if pd.api.types.is_numeric_dtype(pdf[c])][:6]
CATEGORICAL_COLS = ['title', 'genre_primary']
DATE_COLS = [c for c in pdf.columns if pd.api.types.is_datetime64_any_dtype(pdf[c])]

DATE_COL = DATE_COLS[0] if DATE_COLS else None

print("NUMERIC_COLS:", NUMERIC_COLS)
print("CATEGORICAL_COLS:", CATEGORICAL_COLS)
print("DATE_COL:", DATE_COL)

NUMERIC_COLS: ['release_year', 'duration_minutes', 'imdb_rating', 'production_budget', 'box_office_revenue', 'number_of_seasons']
CATEGORICAL_COLS: ['title', 'genre_primary']
DATE_COL: None


## 3) Benchmark utilities

We run each workload multiple times and report mean/median/min.

---


In [27]:
@dataclass
class BenchResult:
    name: str
    runs: List[float]

    @property
    def mean(self): return float(np.mean(self.runs))
    @property
    def median(self): return float(np.median(self.runs))
    @property
    def best(self): return float(np.min(self.runs))
    @property
    def worst(self): return float(np.max(self.runs))

def bench(fn: Callable[[], Any], name: str, repeats: int = 5, warmup: int = 1) -> BenchResult:
    # Warmup
    for _ in range(warmup):
        _ = fn()
        gc.collect()

    runs = []
    for _ in range(repeats):
        gc.collect()
        t0 = time.perf_counter()
        _ = fn()
        t1 = time.perf_counter()
        runs.append(t1 - t0)
    return BenchResult(name=name, runs=runs)

def summarize(results: List[BenchResult]) -> pd.DataFrame:
    return pd.DataFrame({
        "name": [r.name for r in results],
        "mean_s": [r.mean for r in results],
        "median_s": [r.median for r in results],
        "best_s": [r.best for r in results],
        "worst_s": [r.worst for r in results],
        "runs": [r.runs for r in results],
    }).sort_values("mean_s")

## 4) Workload definitions

### Data cleaning (pandas vs polars)

- Fill nulls: numeric columns with median, categorical with "UNKNOWN"
- Drop duplicate rows
- Drop columns that are all-null

### KMeans clustering

- Select numeric columns from cleaned data, scale with StandardScaler, run KMeans

---


In [ ]:
N_CLUSTERS = 5
pldf = pldf_direct  # Use the Polars DataFrame from BigQuery load

# clean data with pandas
def data_cleaning_pandas() -> pd.DataFrame:
    """Data cleaning workload using pandas."""
    df = pdf.copy()
    for c in NUMERIC_COLS:
        if c in df.columns:
            df[c] = df[c].fillna(df[c].median())
    for c in CATEGORICAL_COLS:
        if c in df.columns:
            df[c] = df[c].fillna("UNKNOWN")
    # Drop duplicate rows
    df = df.drop_duplicates()
    # Drop columns that are all null
    df = df.dropna(axis=1, how="all")
    return df

# clean data with polars
def data_cleaning_polars() -> pl.DataFrame:
    """Data cleaning workload using polars."""
    df = pldf.clone()
    exprs = []
    for c in NUMERIC_COLS:
        if c in df.columns:
            median_val = df.select(pl.col(c).median()).item()
            if median_val is None or (isinstance(median_val, float) and np.isnan(median_val)):
                median_val = 0.0
            exprs.append(pl.col(c).fill_null(median_val).fill_nan(median_val).alias(c))
    for c in CATEGORICAL_COLS:
        if c in df.columns:
            exprs.append(pl.col(c).fill_null("UNKNOWN").alias(c))
    if exprs:
        df = df.with_columns(exprs)
    df = df.unique()
    # Drop columns that are all null
    null_cols = [c for c in df.columns if df.select(pl.col(c).is_null().all()).item()]
    if null_cols:
        df = df.drop(null_cols)
    return df

def workload_kmeans_pandas() -> Dict[str, Any]:
    """Data cleaning + KMeans clustering using pandas."""
    df = data_cleaning_pandas()
    avail_num = [c for c in NUMERIC_COLS if c in df.columns]
    X = df[avail_num].to_numpy(dtype=np.float64)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    Xs = StandardScaler(with_mean=True, with_std=True).fit_transform(X)
    km = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=0)
    labels = km.fit_predict(Xs)
    return {"shape": X.shape, "labels_unique": int(np.unique(labels).size)}

def workload_kmeans_polars() -> Dict[str, Any]:
    """Data cleaning + KMeans clustering using polars."""
    df = data_cleaning_polars()
    avail_num = [c for c in NUMERIC_COLS if c in df.columns]
    X = df.select(avail_num).to_numpy().astype(np.float64)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    Xs = StandardScaler(with_mean=True, with_std=True).fit_transform(X)
    km = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=0)
    labels = km.fit_predict(Xs)
    return {"shape": X.shape, "labels_unique": int(np.unique(labels).size)}

## 5) Run benchmarks

Benchmarks:

1. **Data cleaning** – pandas vs polars
2. **KMeans clustering** (data cleaning + scaling + clustering) – pandas vs polars


In [29]:
REPEATS = 5
WARMUP = 1

results = []
# Data cleaning benchmarks
results.append(bench(data_cleaning_pandas, "pandas: data cleaning", repeats=REPEATS, warmup=WARMUP))
results.append(bench(data_cleaning_polars, "polars: data cleaning", repeats=REPEATS, warmup=WARMUP))
# KMeans clustering benchmarks (data cleaning + scale + KMeans)
results.append(bench(workload_kmeans_pandas, "pandas: data cleaning + KMeans", repeats=REPEATS, warmup=WARMUP))
results.append(bench(workload_kmeans_polars, "polars: data cleaning + KMeans", repeats=REPEATS, warmup=WARMUP))

summary = summarize(results)
summary

,name,mean_s,median_s,best_s,worst_s,runs
1,polars: data cleaning,0.003674,0.003479,0.003234,0.004490,"[0.0037211249582469463, 0.00448979192879051, 0..."
0,pandas: data cleaning,0.006448,0.006087,0.005848,0.007807,"[0.007806875044479966, 0.006087167072109878, 0..."
3,polars: data cleaning + KMeans,0.012619,0.012626,0.011488,0.013448,"[0.01344837504439056, 0.012528167106211185, 0...."
2,pandas: data cleaning + KMeans,0.016515,0.015958,0.014327,0.019822,"[0.019822166999801993, 0.015957999974489212, 0..."


In [31]:
# Percent faster: pandas vs polars (from mean_s)
workload_pairs = [
    ("data cleaning", "pandas: data cleaning", "polars: data cleaning"),
    ("data cleaning + KMeans", "pandas: data cleaning + KMeans", "polars: data cleaning + KMeans"),
]
comparisons = []
for label, pd_name, pl_name in workload_pairs:
    pd_row = summary[summary["name"] == pd_name]
    pl_row = summary[summary["name"] == pl_name]
    if len(pd_row) > 0 and len(pl_row) > 0:
        pd_mean = pd_row["mean_s"].iloc[0]
        pl_mean = pl_row["mean_s"].iloc[0]
        if pl_mean < pd_mean:
            pct = (pd_mean - pl_mean) / pd_mean * 100
            faster = "polars"
        else:
            pct = (pl_mean - pd_mean) / pl_mean * 100
            faster = "pandas"
        comparisons.append({"workload": label, "faster": faster, "percent_faster": pct})

comparison_df = pd.DataFrame(comparisons)
comparison_df

,workload,faster,percent_faster
0,data cleaning,polars,43.024033
1,data cleaning + KMeans,polars,23.590426
